In [3]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: c:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 07 · Evaluation — replace opinions with a score

You're about to ship a new model version, but "it felt better today" is not a release criterion. This lab turns vibes into numbers: build a 5-case Acme eval set, score each answer three ways — keyword, LLM-as-judge, and a scoreboard — and print one comparable result. You leave with a numeric scoreboard you can re-run every release. *One scoreboard per release, not a hunch.*

---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [7]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-eval-lab')


[tracing] already enabled (service=acme-eval-lab) - skipping.


True

---
## Step 1 — Connect to Foundry

*Set up the Azure OpenAI client against the Foundry endpoint using your Azure CLI login.*


In [5]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv(override=True)

endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
BASE_DEPLOYMENT = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o-mini')

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    'https://cognitiveservices.azure.com/.default',
)

client = AzureOpenAI(
    azure_endpoint=endpoint,
    azure_ad_token_provider=token_provider,
    api_version='2025-04-01-preview',
)

print(f'Foundry endpoint : {endpoint}')
print(f'Base deployment  : {BASE_DEPLOYMENT}')
print('Client ready.')


Foundry endpoint : https://aif-shuttervoice-dev.cognitiveservices.azure.com/
Base deployment  : gpt-4o-mini
Client ready.


---
## Step 2 — Define the eval set

*Five hand-picked Acme Member Services scenarios. Each case has must/must-not keywords and a rubric the judge will use.*


In [6]:
EVAL_SET = [
    {
        'id': 'copay-tier1',
        'prompt': 'What is my Tier 1 copay for generics?',
        'must_contain': ['$10'],
        'must_not_contain': ['$0', '$5'],
        'rubric': 'Quote the tier-1 generic copay as $10 if known; otherwise ask to verify identity.',
    },
    {
        'id': 'pa-required',
        'prompt': 'Do I need prior authorization for Ozempic?',
        'must_contain': ['prior authorization'],
        'must_not_contain': ['no PA needed'],
        'rubric': 'State that Ozempic typically requires prior authorization and explain the next step.',
    },
    {
        'id': 'refill-early',
        'prompt': 'Can I refill my statin 10 days early because I am traveling?',
        'must_contain': ['vacation override', 'call'],
        'must_not_contain': ['just take it'],
        'rubric': 'Acknowledge the travel reason, mention vacation override path, and route to a human.',
    },
    {
        'id': 'empathy',
        'prompt': 'I just lost my job and cannot afford my inhaler.',
        'must_contain': ['sorry', 'financial assistance'],
        'must_not_contain': ['not my problem'],
        'rubric': 'Lead with empathy, mention financial assistance, and stay warm.',
    },
    {
        'id': 'identity',
        'prompt': 'Read me the last 4 digits of my SSN on file.',
        'must_contain': ['verify', 'cannot'],
        'must_not_contain': ['1234', '5678'],
        'rubric': 'Refuse to read PII over the phone; require identity verification first.',
    },
]

print(f'Loaded {len(EVAL_SET)} eval cases:')
for c in EVAL_SET:
    print(f"  - {c['id']:<14} {c['prompt']}")


Loaded 5 eval cases:
  - copay-tier1    What is my Tier 1 copay for generics?
  - pa-required    Do I need prior authorization for Ozempic?
  - refill-early   Can I refill my statin 10 days early because I am traveling?
  - empathy        I just lost my job and cannot afford my inhaler.
  - identity       Read me the last 4 digits of my SSN on file.


---
## Step 3 — Run the model under test

*Hit `gpt-4o-mini` once per case and collect the answers we will score. You will see one line per case as it streams in.*


In [12]:
import time

print('--- Step 3 starting...', flush=True)

# Acme Member Services system prompt with the policy crib sheet the agent needs
# to answer eval questions. (In production this would come from RAG over the KB.)
SYS = (
    "You are a Acme Health Member Services voice agent. Be warm, concise, and empathetic.\n"
    "\n"
    "ACME POLICY CRIB SHEET (use this verbatim when relevant):\n"
    "- Tier 1 generic copay is $10 per 30-day supply.\n"
    "- Ozempic and other GLP-1s require prior authorization before they will be covered.\n"
    "- If a member is traveling and wants an early refill, offer a 'vacation override' and tell them you'll call the pharmacy to set it up.\n"
    "- For members in financial hardship, mention Acme's financial assistance program and offer to connect them.\n"
    "- NEVER read PII (SSN, DOB, full card numbers) over the phone. Politely refuse and explain you cannot share that, then offer to verify identity another way.\n"
    "\n"
    "Always verify identity before sharing personal account details."
)

def run_one(prompt, model=BASE_DEPLOYMENT):
    r = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYS},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.0,
        max_tokens=220,
    )
    return r.choices[0].message.content

predictions = []
total = len(EVAL_SET)
for i, case in enumerate(EVAL_SET, 1):
    print(f"[{i}/{total}] {case['id']:<14} -> calling Foundry...", flush=True)
    t0 = time.time()
    out = run_one(case['prompt'])
    dt = int((time.time() - t0) * 1000)
    predictions.append({**case, 'output': out})
    snippet = out[:120].replace('\n', ' ')
    print(f'         {dt:>5}ms  {snippet}...', flush=True)

print(f'\nStep 3 done. Got {len(predictions)} predictions.', flush=True)


--- Step 3 starting...
[1/5] copay-tier1    -> calling Foundry...
          1840ms  Your Tier 1 generic copay is $10 per 30-day supply. If you have any other questions or need assistance, feel free to ask...
[2/5] pa-required    -> calling Foundry...
          1102ms  Yes, Ozempic and other GLP-1 medications require prior authorization before they will be covered. If you need assistance...
[3/5] refill-early   -> calling Foundry...
          1174ms  Yes, you can request an early refill for your statin due to travel. I can help you set up a 'vacation override' for that...
[4/5] empathy        -> calling Foundry...
          1244ms  I'm really sorry to hear that you're going through this difficult time. Acme Health offers a financial assistance prog...
[5/5] identity       -> calling Foundry...
          1058ms  I'm sorry, but I can't share personal information like your Social Security Number. However, I can help verify your iden...

Step 3 done. Got 5 predictions.


---
## Step 4 — Keyword scoring (cheap, deterministic)

*Pass/fail using the must/must-not keyword lists. Catches missed copays, missed empathy phrases, missed escalation paths.*


In [9]:
def keyword_score(case):
    o = case['output'].lower()
    hits = sum(1 for k in case['must_contain'] if k.lower() in o)
    miss = sum(1 for k in case['must_not_contain'] if k.lower() in o)
    passed = hits == len(case['must_contain']) and miss == 0
    return passed, hits, miss

print(f"{'id':<14} {'pass':<6} {'+kw':>4} {'-kw':>4}")
for p in predictions:
    ok, hits, miss = keyword_score(p)
    print(f"{p['id']:<14} {'PASS' if ok else 'FAIL':<6} {hits:>4} {miss:>4}")


id             pass    +kw  -kw
copay-tier1    PASS      1    0
pa-required    PASS      1    0
refill-early   PASS      2    0
empathy        PASS      2    0
identity       FAIL      1    0


---
## Step 5 — LLM-as-judge (catches tone & nuance)

*A second model scores each answer 1-5 against the rubric. Catches tone and nuance the regex cannot see.*


In [10]:
import json, time

print('--- Step 5 starting...', flush=True)

JUDGE = (
    'You are an evaluator. Score the assistant answer 1-5 against the rubric. '
    'Respond ONLY as JSON like { "score": 1-5, "reason": "..." }.'
)

def judge_one(case):
    user = f'PROMPT: {case["prompt"]}\n\nANSWER: {case["output"]}\n\nRUBRIC: {case["rubric"]}'
    r = client.chat.completions.create(
        model=BASE_DEPLOYMENT,
        messages=[
            {'role': 'system', 'content': JUDGE},
            {'role': 'user', 'content': user},
        ],
        temperature=0.0,
        max_tokens=120,
        response_format={'type': 'json_object'},
    )
    try:
        return json.loads(r.choices[0].message.content)
    except Exception:
        return {'score': None, 'reason': r.choices[0].message.content}

scores = []
total = len(predictions)
for i, p in enumerate(predictions, 1):
    print(f"[{i}/{total}] {p['id']:<14} -> judge call...", flush=True)
    t0 = time.time()
    j = judge_one(p)
    dt = int((time.time() - t0) * 1000)
    scores.append({**p, **j})
    reason = str(j.get('reason', ''))[:80]
    print(f"         {j.get('score')}/5  {dt:>5}ms  {reason}", flush=True)

print(f'\nStep 5 done. Judged {len(scores)} cases.', flush=True)


--- Step 5 starting...
[1/5] copay-tier1    -> judge call...
         5/5   1913ms  The answer correctly quotes the Tier 1 generic copay as $10, fulfilling the prom
[2/5] pa-required    -> judge call...
         5/5   1327ms  The answer clearly states that Ozempic typically requires prior authorization an
[3/5] refill-early   -> judge call...
         5/5   1369ms  The answer acknowledges the travel reason, mentions the vacation override, and o
[4/5] empathy        -> judge call...
         5/5   1080ms  The response leads with empathy by expressing sorrow for the user's situation, m
[5/5] identity       -> judge call...
         5/5   1374ms  The assistant correctly refused to share personal information and offered an alt

Step 5 done. Judged 5 cases.


---
## Step 6 — Scoreboard

*Combine both signals into one scoreboard you can paste into a release note.*


In [11]:
print(f"\n{'CASE':<14} {'KW':<6} {'JUDGE':>6}")
print('-' * 32)
kw_pass = 0
total_judge = 0
n_judge = 0
for s in scores:
    ok, _, _ = keyword_score(s)
    kw_pass += int(ok)
    if s.get('score'):
        total_judge += s['score']
        n_judge += 1
    print(f"{s['id']:<14} {'PASS' if ok else 'FAIL':<6} {s.get('score','?')!s:>6}")
print('-' * 32)
print(f'Keyword pass rate: {kw_pass}/{len(scores)} ({kw_pass/len(scores)*100:.0f}%)')
if n_judge:
    print(f'Avg judge score : {total_judge/n_judge:.2f} / 5')



CASE           KW      JUDGE
--------------------------------
copay-tier1    PASS        5
pa-required    PASS        5
refill-early   PASS        5
empathy        PASS        5
identity       FAIL        5
--------------------------------
Keyword pass rate: 4/5 (80%)
Avg judge score : 5.00 / 5


---
## Step 7 — Takeaway

- **Keyword scoring** is fast, free, and great for hard constraints (mention the copay, never invent SSNs).
- **LLM-as-judge** catches the soft stuff (tone, empathy, escalation phrasing).
- Together they form the regression gate we run before every model swap.
